03_build_gold — KPI comerciales y logísticos
Las 7 preguntas de negocio
1. ¿Cómo evolucionan las ventas mensuales? → gold_monthly_sales.
2. ¿Qué porcentaje representan las categorías de las ventas mensuales? → gold_monthly_category_share.
3. ¿Cuál es el ticket promedio de cada mes? → gold_monthly_sales.
4. ¿Cuáles son los 10 productos más vendidos por mes? → gold_monthly_top_products.
5. ¿Qué relación existe entre categorías en portugués y estados de destino? → gold_monthly_category_state.
6. ¿Cómo se comportan las entregas y los atrasos por estado? → gold_delivery_performance.
7. ¿Cómo se usan los medios de pago? → gold_monthly_payment_methods.
8. ¿Como fue desarrollandose mi portfolio a lo largo del tiempo? gold_monthly_portfolio_evolution

Definiciones acordadas
- Solo pedidos con order_status = 'delivered'; se atribuyen al mes de compra, no al mes de entrega.
- GMV comercial (BRL): suma de order_items.price (sin flete). Flete (BRL): suma de freight_value. Valor de artículos + flete: suma de ambos. Son importes brutos del dataset, no ingresos netos contables (no tenemos devoluciones/reembolsos completos).
- Ticket promedio: GMV dividido por cantidad de pedidos entregados que tengan al menos un ítem. Los pedidos entregados sin ítems se registran aparte.
- Unidades de producto: número de filas de order_items (cada fila equivale a una unidad vendida). No hay nombres comerciales de productos en Olist; el top se identifica por product_id.
- Geografía: customer_state (destino del comprador). No se usa un INNER JOIN con silver_geolocation, para no descartar clientes cuyo ZIP carece de correspondencia.
- Entrega evaluable: pedido entregado con fechas de compra, entrega real y entrega estimada, con fecha_inconsistente = false y entrega no anterior a compra. La bandera is_delivered_late de Silver determina el atraso.
- Pagos: payment_value se calcula directamente desde silver_order_payments, sin cruzarlo con cada ítem. Una orden puede emplear más de un método; el porcentaje de órdenes usuarias por método puede sumar más de 100%.
- Reejecución: recomputamos los agregados desde Silver y hacemos MERGE por clave Gold; borramos grupos que desaparezcan de la fuente. No se modifica Bronze ni Silver.

In [0]:
#Configuración y comprobaciones de fuentes
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable
from functools import reduce

CATALOG = "workspace"
SCHEMA = "olist_lakehouse"
PREFIX = f"{CATALOG}.{SCHEMA}"

spark.conf.set("spark.sql.session.timeZone", "UTC")

T = {
    "orders": f"{PREFIX}.silver_orders",
    "items": f"{PREFIX}.silver_order_items",
    "payments": f"{PREFIX}.silver_order_payments",
    "customers": f"{PREFIX}.silver_customers",
    "products": f"{PREFIX}.silver_products",
}

required = {
    "orders": {"order_id", "customer_id", "order_status", "order_purchase_timestamp",
               "order_delivered_customer_date", "order_estimated_delivery_date",
               "fecha_inconsistente", "is_delivered_late"},
    "items": {"order_id", "order_item_id", "product_id", "price", "freight_value"},
    "payments": {"order_id", "payment_sequential", "payment_type", "payment_value", "payment_installments"},
    "customers": {"customer_id", "customer_state"},
    "products": {"product_id", "product_category_name", "category_for_analysis"},
}

for name, table in T.items():
    df = spark.table(table)
    missing = sorted(required[name] - set(df.columns))
    if missing:
        raise ValueError(f"Faltan columnas en {table}: {missing}")
    print(f"OK fuente: {table}")

# La capa Gold no debe ocultar importes faltantes o negativos.
items_error = spark.table(T["items"]).filter(
    F.col("price").isNull() | (F.col("price") < 0) |
    F.col("freight_value").isNull() | (F.col("freight_value") < 0)
).count()
if items_error:
    raise ValueError(f"Hay {items_error} líneas con precio/flete nulo o negativo. Revisar Silver.")

print("OK: importes de items válidos. Huso horario:", spark.conf.get("spark.sql.session.timeZone"))


In [0]:
#Celda 2 — Modelo intermedio seguro (una fila por pedido y por ítem)
#Creamos DataFrames temporales en memoria lógica de Spark, no tablas permanentes adicionales. Los pagos se agregan a nivel pedido antes de cualquier join con las órdenes. Las líneas comerciales solo se cruzan con dimensiones 1:1.

orders = (
    spark.table(T["orders"])
    .filter(F.lower(F.trim(F.col("order_status"))) == "delivered")
    .withColumn("mes", F.to_date(F.date_trunc("month", F.col("order_purchase_timestamp"))))
    .filter(F.col("mes").isNotNull())
    .select("order_id", "customer_id", "mes", "order_purchase_timestamp",
            "order_delivered_customer_date", "order_estimated_delivery_date",
            "fecha_inconsistente", "is_delivered_late")
)

customers = (
    spark.table(T["customers"])
    .select("customer_id", F.upper(F.trim(F.col("customer_state"))).alias("estado_origen"))
)

orders_geo = (
    orders.join(customers, on="customer_id", how="left")
    .withColumn("estado", F.coalesce(
        F.when(F.col("estado_origen") != "", F.col("estado_origen")),
        F.lit("DESCONOCIDO")
    ))
    .drop("estado_origen")
)

items = spark.table(T["items"]).select(
    "order_id", "order_item_id", "product_id", "price", "freight_value"
)
products = spark.table(T["products"]).select(
    "product_id", "product_category_name", "category_for_analysis"
)

# Una fila por ítem vendido, nunca unir con pagos sin agregarlos.
items_fact = (
    items.join(orders_geo.select("order_id", "mes", "estado"), "order_id", "inner")
    .join(products, "product_id", "left")
    .withColumn("categoria_analitica", F.coalesce(
        F.col("category_for_analysis"), F.lit("Sin categoría")
    ))
    .withColumn("categoria_pt", F.coalesce(
        F.col("product_category_name"), F.lit("Sem categoria")
    ))
)

items_by_order = (
    items.groupBy("order_id")
    .agg(
        F.count("*").alias("cantidad_items"),
        F.sum("price").alias("gmv_pedido_brl"),
        F.sum("freight_value").alias("flete_pedido_brl")
    )
)

payments = spark.table(T["payments"]).select(
    "order_id", "payment_sequential", "payment_type", "payment_value", "payment_installments"
)

payments_by_order = payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("pago_pedido_brl"),
    F.count("*").alias("cantidad_registros_pago")
)

order_fact = (
    orders_geo
    .join(items_by_order, "order_id", "left")
    .join(payments_by_order, "order_id", "left")
    .withColumn("cantidad_items", F.coalesce(F.col("cantidad_items"), F.lit(0)))
    .withColumn("gmv_pedido_brl", F.coalesce(F.col("gmv_pedido_brl"), F.lit(0).cast("decimal(18,2)")))
    .withColumn("flete_pedido_brl", F.coalesce(F.col("flete_pedido_brl"), F.lit(0).cast("decimal(18,2)")))
    .withColumn("cantidad_registros_pago", F.coalesce(F.col("cantidad_registros_pago"), F.lit(0)))
)

print("Pedidos entregados con mes válido:", order_fact.count())
print("Ítems en pedidos entregados:", items_fact.count())
print("Nota: el pago puede diferir de GMV+flete; se reporta por separado.")


In [0]:
#Celda 3 — Constructor Gold reproducible
#Implementación Delta MERGE: inserta nuevas claves, actualiza solo contenido diferente y elimina agregados obsoletos. Cada tabla contiene gold_row_hash para verificar que una reejecución no modifique innecesariamente filas idénticas.
def with_content_hash(df):
    if "gold_row_hash" in df.columns:
        return df  # portfolio ya calcula un hash estable
    columnas = sorted(df.columns)
    return df.withColumn(
        "gold_row_hash",
        F.sha2(
            F.to_json(F.struct(*[F.col(c) for c in columnas]),
                      options={"ignoreNullFields": "false"}),
            256
        )
    )


def publish_gold(df, table_short_name, grain_cols):
    destination = f"{PREFIX}.{table_short_name}"
    result = with_content_hash(df)
    if result.filter(reduce(lambda a, b: a | b,
                            [F.col(c).isNull() for c in grain_cols])).limit(1).count():
        raise ValueError(f"Clave Gold nula en {destination}")
    if result.groupBy(*grain_cols).count().filter("count > 1").limit(1).count():
        raise ValueError(f"Grano duplicado en {destination}: {grain_cols}")

    # Crear tabla vacía una sola vez. No sobrescribe tablas Silver/Bronze.
    if not spark.catalog.tableExists(destination):
        result.limit(0).write.format("delta").mode("append").saveAsTable(destination)

    # Si la tabla Gold ya existía con otro esquema, detener y revisarlo explícitamente.
    actual = set(spark.table(destination).columns)
    expected = set(result.columns)
    if actual != expected:
        raise ValueError(
            f"Esquema diferente para {destination}. "
            f"Faltan en destino: {expected - actual}; sobran: {actual - expected}"
        )

    join_condition = " AND ".join([f"t.`{c}` <=> s.`{c}`" for c in grain_cols])
    (
        DeltaTable.forName(spark, destination).alias("t")
        .merge(result.alias("s"), join_condition)
        .whenMatchedUpdateAll(condition="NOT (t.gold_row_hash <=> s.gold_row_hash)")
        .whenNotMatchedInsertAll()
        .whenNotMatchedBySourceDelete()
        .execute()
    )
    print(f"PUBLICADO: {destination} | grano: {', '.join(grain_cols)}")


In [0]:
#Ventas mensuales y ticket promedio
gold_monthly_sales = (
    order_fact.groupBy("mes")
    .agg(
        F.count("*").alias("pedidos_entregados"),
        F.sum(F.when(F.col("cantidad_items") > 0, 1).otherwise(0)).alias("pedidos_con_items"),
        F.sum(F.when(F.col("cantidad_items") == 0, 1).otherwise(0)).alias("pedidos_sin_items"),
        F.sum("cantidad_items").alias("unidades_vendidas"),
        F.sum("gmv_pedido_brl").alias("gmv_brl"),
        F.sum("flete_pedido_brl").alias("flete_brl"),
        F.sum("pago_pedido_brl").alias("pagos_brl"),
        F.sum(F.when(F.col("pago_pedido_brl").isNotNull(), 1).otherwise(0)).alias("pedidos_con_pago")
    )
    .withColumn("valor_items_mas_flete_brl", F.col("gmv_brl") + F.col("flete_brl"))
    .withColumn(
        "ticket_promedio_brl",
        F.when(F.col("pedidos_con_items") > 0,
               (F.col("gmv_brl") / F.col("pedidos_con_items")).cast("decimal(18,2)"))
    )
    .withColumn("anio", F.year("mes"))
    .withColumn("mes_numero", F.month("mes"))
)

w_mes = Window.orderBy("mes")
gold_monthly_sales = (
    gold_monthly_sales
    .withColumn("gmv_mes_anterior_brl", F.lag("gmv_brl").over(w_mes))
    .withColumn("variacion_gmv_mensual_pct", F.when(
        F.col("gmv_mes_anterior_brl") > 0,
        F.round(100.0 * (
            F.col("gmv_brl") - F.col("gmv_mes_anterior_brl")
        ) / F.col("gmv_mes_anterior_brl"), 2)
    ))
)

publish_gold(gold_monthly_sales, "gold_monthly_sales", ["mes"])
display(spark.table(f"{PREFIX}.gold_monthly_sales").orderBy("mes"))


In [0]:
#Participación mensual por categoría
category_month_base = (
    items_fact.groupBy("mes", "categoria_analitica")
    .agg(
        F.count("*").alias("unidades_vendidas"),
        F.countDistinct("order_id").alias("pedidos_con_categoria"),
        F.sum("price").alias("gmv_brl"),
        F.sum("freight_value").alias("flete_brl")
    )
)

w_cat_mes = Window.partitionBy("mes")
gold_monthly_category_share = (
    category_month_base
    .withColumn("gmv_total_mes_brl", F.sum("gmv_brl").over(w_cat_mes))
    .withColumn("participacion_gmv_pct", F.when(
        F.col("gmv_total_mes_brl") > 0,
        F.round(F.col("gmv_brl") * 100.0 / F.col("gmv_total_mes_brl"), 4)
    ))
)

publish_gold(gold_monthly_category_share, "gold_monthly_category_share",
             ["mes", "categoria_analitica"])
display(spark.table(f"{PREFIX}.gold_monthly_category_share")
        .orderBy(F.desc("mes"), F.desc("gmv_brl")).limit(30))


In [0]:
#Top 10 productos vendidos por mes
product_month_base = (
    items_fact.groupBy("mes", "product_id", "categoria_pt")
    .agg(
        F.count("*").alias("unidades_vendidas"),
        F.countDistinct("order_id").alias("pedidos_con_producto"),
        F.sum("price").alias("gmv_brl")
    )
)

w_top = Window.partitionBy("mes").orderBy(
    F.desc("unidades_vendidas"), F.desc("gmv_brl"), F.asc("product_id")
)
gold_monthly_top_products = (
    product_month_base
    .withColumn("ranking_mes", F.row_number().over(w_top))
    .filter(F.col("ranking_mes") <= 10)
)

publish_gold(gold_monthly_top_products, "gold_monthly_top_products",
             ["mes", "ranking_mes"])
display(spark.table(f"{PREFIX}.gold_monthly_top_products")
        .orderBy(F.desc("mes"), "ranking_mes").limit(30))


In [0]:
#Relación entre categorías en portugués y estados

cat_state_base = (
    items_fact.groupBy("mes", "estado", "categoria_pt")
    .agg(
        F.count("*").alias("unidades_vendidas"),
        F.countDistinct("order_id").alias("pedidos_con_categoria"),
        F.sum("price").alias("gmv_brl")
    )
)

w_estado = Window.partitionBy("mes", "estado")
w_categoria = Window.partitionBy("mes", "categoria_pt")
gold_monthly_category_state = (
    cat_state_base
    .withColumn("gmv_total_estado_mes_brl", F.sum("gmv_brl").over(w_estado))
    .withColumn("gmv_total_categoria_mes_brl", F.sum("gmv_brl").over(w_categoria))
    .withColumn("participacion_categoria_en_estado_pct", F.when(
        F.col("gmv_total_estado_mes_brl") > 0,
        F.round(100.0 * F.col("gmv_brl") / F.col("gmv_total_estado_mes_brl"), 4)
    ))
    .withColumn("participacion_estado_en_categoria_pct", F.when(
        F.col("gmv_total_categoria_mes_brl") > 0,
        F.round(100.0 * F.col("gmv_brl") / F.col("gmv_total_categoria_mes_brl"), 4)
    ))
)

publish_gold(gold_monthly_category_state, "gold_monthly_category_state",
             ["mes", "estado", "categoria_pt"])
display(spark.table(f"{PREFIX}.gold_monthly_category_state")
        .orderBy(F.desc("mes"), F.desc("gmv_brl")).limit(30))


In [0]:
#Indicadores de entrega: Brasil y estados

delivery = (
    orders_geo
    .withColumn("tiene_fechas", (
        F.col("order_purchase_timestamp").isNotNull() &
        F.col("order_delivered_customer_date").isNotNull() &
        F.col("order_estimated_delivery_date").isNotNull()
    ))
    .withColumn("fechas_consistentes", (
        ~F.coalesce(F.col("fecha_inconsistente"), F.lit(False)) &
        (F.col("order_delivered_customer_date") >= F.col("order_purchase_timestamp"))
    ))
    .withColumn("es_evaluable", F.col("tiene_fechas") & F.col("fechas_consistentes"))
    .withColumn("es_tardia", F.when(F.col("es_evaluable"),
                                    F.coalesce(F.col("is_delivered_late"),
                                        F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"))))
    .withColumn("dias_entrega", F.when(F.col("es_evaluable"),
        F.datediff(F.to_date("order_delivered_customer_date"), F.to_date("order_purchase_timestamp"))))
    .withColumn("dias_atraso", F.when(F.col("es_evaluable") & F.col("es_tardia"),
        F.greatest(F.datediff(F.to_date("order_delivered_customer_date"),
                             F.to_date("order_estimated_delivery_date")), F.lit(0))))
)


def aggregate_delivery(df, dimensions):
    return (
        df.groupBy(*dimensions)
        .agg(
            F.count("*").alias("pedidos_entregados"),
            F.sum(F.when(F.col("es_evaluable"), 1).otherwise(0)).alias("entregas_evaluables"),
            F.sum(F.when(F.col("es_evaluable") & F.col("es_tardia"), 1).otherwise(0)).alias("entregas_tardias"),
            F.sum(F.when(~F.col("tiene_fechas"), 1).otherwise(0)).alias("entregas_con_fechas_faltantes"),
            F.sum(F.when(F.col("tiene_fechas") & ~F.col("fechas_consistentes"), 1).otherwise(0)).alias("entregas_fechas_inconsistentes"),
            F.round(F.avg("dias_entrega"), 2).alias("dias_promedio_entrega"),
            F.round(F.avg("dias_atraso"), 2).alias("dias_promedio_atraso_si_tardia")
        )
        .withColumn("porcentaje_tardias", F.when(
            F.col("entregas_evaluables") > 0,
            F.round(F.col("entregas_tardias") * 100.0 / F.col("entregas_evaluables"), 2)
        ))
        .withColumn("porcentaje_a_tiempo", F.when(
            F.col("entregas_evaluables") > 0,
            F.round((F.col("entregas_evaluables") - F.col("entregas_tardias")) * 100.0 /
                    F.col("entregas_evaluables"), 2)
        ))
    )

state_delivery = (
    aggregate_delivery(delivery, ["mes", "estado"])
    .withColumn("nivel_geografico", F.lit("ESTADO"))
)
br_delivery = (
    aggregate_delivery(delivery, ["mes"])
    .withColumn("estado", F.lit("BR_TOTAL"))
    .withColumn("nivel_geografico", F.lit("BRASIL"))
)

gold_delivery_performance = state_delivery.unionByName(br_delivery)
publish_gold(gold_delivery_performance, "gold_delivery_performance",
             ["mes", "nivel_geografico", "estado"])
display(spark.table(f"{PREFIX}.gold_delivery_performance")
        .filter("nivel_geografico = 'BRASIL'").orderBy("mes"))


In [0]:
#Métodos de pago
payment_fact = (
    payments
    .join(orders_geo.select("order_id", "mes"), "order_id", "inner")
    .withColumn("metodo_pago", F.coalesce(
        F.when(F.trim(F.col("payment_type")) != "", F.trim(F.col("payment_type"))),
        F.lit("unknown")
    ))
)

payment_group = (
    payment_fact.groupBy("mes", "metodo_pago")
    .agg(
        F.count("*").alias("registros_pago"),
        F.countDistinct("order_id").alias("pedidos_que_usan_metodo"),
        F.sum("payment_value").alias("valor_pagado_brl"),
        F.round(F.avg(F.when(
            F.col("metodo_pago") == "credit_card", F.col("payment_installments")
        )), 2).alias("cuotas_promedio_credito")
    )
)

payments_total_orders = (
    payment_fact.groupBy("mes")
    .agg(F.countDistinct("order_id").alias("pedidos_con_algun_pago"))
)

w_pay_mes = Window.partitionBy("mes")
gold_monthly_payment_methods = (
    payment_group.join(payments_total_orders, "mes", "left")
    .withColumn("valor_pagado_total_mes_brl", F.sum("valor_pagado_brl").over(w_pay_mes))
    .withColumn("porcentaje_valor_pagado", F.when(
        F.col("valor_pagado_total_mes_brl") > 0,
        F.round(100.0 * F.col("valor_pagado_brl") / F.col("valor_pagado_total_mes_brl"), 4)
    ))
    .withColumn("porcentaje_pedidos_usuarios", F.when(
        F.col("pedidos_con_algun_pago") > 0,
        F.round(100.0 * F.col("pedidos_que_usan_metodo") / F.col("pedidos_con_algun_pago"), 4)
    ))
)

publish_gold(gold_monthly_payment_methods, "gold_monthly_payment_methods",
             ["mes", "metodo_pago"])
display(spark.table(f"{PREFIX}.gold_monthly_payment_methods")
        .orderBy(F.desc("mes"), F.desc("valor_pagado_brl")).limit(30))


In [0]:
#Validaciones integrales. Verificamos que Gold reconcilie con sus hechos base. Si falla algún assert, no dar por terminado el notebook. La comparación usa decimales de BRL redondeados a centavos y montos totalizados por mes.
def compare_by_month(left_df, right_df, left_col, right_col, label):
    l = left_df.select("mes", F.round(F.col(left_col), 2).alias("v_left"))
    r = right_df.select("mes", F.round(F.col(right_col), 2).alias("v_right"))
    bad = (l.join(r, "mes", "full")
           .filter(~F.col("v_left").eqNullSafe(F.col("v_right"))))
    n = bad.count()
    if n:
        display(bad.orderBy("mes"))
        raise AssertionError(f"Descuadre {label}: {n} meses")
    print(f"OK: {label}")

sales = spark.table(f"{PREFIX}.gold_monthly_sales")
category = spark.table(f"{PREFIX}.gold_monthly_category_share")
cat_state = spark.table(f"{PREFIX}.gold_monthly_category_state")
top = spark.table(f"{PREFIX}.gold_monthly_top_products")
delivery_gold = spark.table(f"{PREFIX}.gold_delivery_performance")
payment_gold = spark.table(f"{PREFIX}.gold_monthly_payment_methods")

compare_by_month(
    sales,
    category.groupBy("mes").agg(F.sum("gmv_brl").alias("gmv_cat")),
    "gmv_brl", "gmv_cat", "ventas = suma de categorías"
)
compare_by_month(
    sales,
    cat_state.groupBy("mes").agg(F.sum("gmv_brl").alias("gmv_estados")),
    "gmv_brl", "gmv_estados", "ventas = categorías por estado"
)

compare_by_month(
    sales.select("mes", "pagos_brl"),
    payment_gold.groupBy("mes").agg(F.sum("valor_pagado_brl").alias("pagos_metodos")),
    "pagos_brl", "pagos_metodos", "pagos por pedido = pagos por método"
)

compare_by_month(
    sales.select("mes", "pedidos_entregados"),
    delivery_gold.filter("nivel_geografico = 'BRASIL'").select("mes", "pedidos_entregados"),
    "pedidos_entregados", "pedidos_entregados", "entregas Brasil = pedidos entregados"
)

# Cobertura del top 10 por mes (si existen >=10 productos en ese mes, 10 filas).
top_bad = (top.groupBy("mes").agg(F.count("*").alias("n"))
           .filter((F.col("n") > 10) | (F.col("n") < 1)).count())
assert top_bad == 0, "Top 10 fuera de rango"
assert top.filter((F.col("ranking_mes") < 1) | (F.col("ranking_mes") > 10)).count() == 0

# Denominadores logísticos y consistencia porcentual.
assert delivery_gold.filter(
    (F.col("entregas_tardias") > F.col("entregas_evaluables")) |
    (F.col("entregas_evaluables") > F.col("pedidos_entregados")) |
    (F.col("entregas_con_fechas_faltantes") + F.col("entregas_fechas_inconsistentes")
     + F.col("entregas_evaluables") != F.col("pedidos_entregados"))
).count() == 0, "Inconsistencia en entrega evaluable / excluida"

for table in ["gold_monthly_sales", "gold_monthly_category_share", "gold_monthly_top_products",
              "gold_monthly_category_state", "gold_delivery_performance", "gold_monthly_payment_methods"]:
    print(table, spark.table(f"{PREFIX}.{table}").count(), "filas")
print("VALIDACIONES GOLD SUPERADAS")


In [0]:
#Consultas para el análisis y gráficos Databricks

# Modificá el mes para analizar otro corte.
MES_EJEMPLO = "2018-01-01"

print("EVOLUCIÓN / TICKET PROMEDIO")
display(spark.sql(f"""
SELECT mes, pedidos_con_items, gmv_brl, ticket_promedio_brl, variacion_gmv_mensual_pct
FROM {PREFIX}.gold_monthly_sales
ORDER BY mes
"""))

print("CATEGORÍAS PORCENTUALES DEL MES")
display(spark.sql(f"""
SELECT mes, categoria_analitica, gmv_brl, participacion_gmv_pct
FROM {PREFIX}.gold_monthly_category_share
WHERE mes = DATE '{MES_EJEMPLO}'
ORDER BY participacion_gmv_pct DESC
"""))

print("TOP 10 PRODUCTOS DEL MES")
display(spark.sql(f"""
SELECT mes, ranking_mes, product_id, categoria_pt, unidades_vendidas, gmv_brl
FROM {PREFIX}.gold_monthly_top_products
WHERE mes = DATE '{MES_EJEMPLO}'
ORDER BY ranking_mes
"""))

print("CATEGORÍAS EN PORTUGUÉS POR ESTADO")
display(spark.sql(f"""
SELECT estado, categoria_pt, unidades_vendidas, gmv_brl,
       participacion_categoria_en_estado_pct,
       participacion_estado_en_categoria_pct
FROM {PREFIX}.gold_monthly_category_state
WHERE mes = DATE '{MES_EJEMPLO}'
ORDER BY gmv_brl DESC
LIMIT 30
"""))

print("ENTREGAS POR ESTADO")
display(spark.sql(f"""
SELECT estado, entregas_evaluables, entregas_tardias,
       porcentaje_tardias, dias_promedio_entrega
FROM {PREFIX}.gold_delivery_performance
WHERE mes = DATE '{MES_EJEMPLO}' AND nivel_geografico = 'ESTADO'
ORDER BY porcentaje_tardias DESC NULLS LAST
"""))

print("USO DE MÉTODOS DE PAGO")
display(spark.sql(f"""
SELECT metodo_pago, registros_pago, pedidos_que_usan_metodo,
       valor_pagado_brl, porcentaje_valor_pagado, porcentaje_pedidos_usuarios
FROM {PREFIX}.gold_monthly_payment_methods
WHERE mes = DATE '{MES_EJEMPLO}'
ORDER BY valor_pagado_brl DESC
"""))


In [0]:
#Construye gold_monthly_portfolio_evolution. Mide como fue desarrollandose mi portfolio a lo largo del tiempo
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

BASE = "workspace.olist_lakehouse"

SOURCE = f"{BASE}.gold_monthly_category_share"
TARGET = f"{BASE}.gold_monthly_portfolio_evolution"

# 1. Fuente: una fila por mes y categoría
categorias = (
    spark.table(SOURCE)
    .groupBy("mes", "categoria_analitica")
    .agg(
        F.sum("unidades_vendidas").alias("unidades_vendidas"),
        F.sum("gmv_brl").alias("gmv_categoria_brl")
    )
)

# 2. Ventanas por categoría y por mes
w_categoria = Window.partitionBy("categoria_analitica")
w_historial = Window.partitionBy(
    "categoria_analitica"
).orderBy("mes")

w_mes = Window.partitionBy("mes")
w_ranking = Window.partitionBy("mes").orderBy(
    F.desc("gmv_categoria_brl"),
    F.asc("categoria_analitica")
)

# 3. Primera aparición y última venta anterior
historico = (
    categorias
    .withColumn(
        "primer_mes_venta",
        F.min("mes").over(w_categoria)
    )
    .withColumn(
        "mes_venta_anterior",
        F.lag("mes").over(w_historial)
    )
    .withColumn(
        "gmv_total_mes",
        F.sum("gmv_categoria_brl").over(w_mes)
    )
    .withColumn(
        "ranking_categoria",
        F.row_number().over(w_ranking)
    )
)

# 4. Indicadores por mes
portfolio = (
    historico
    .groupBy("mes")
    .agg(
        F.round(F.max("gmv_total_mes"), 2).alias("gmv_brl"),

        F.sum("unidades_vendidas").alias("unidades_vendidas"),

        F.countDistinct(
            F.when(
                (F.col("categoria_analitica") != "Sin categoría") &
                (F.col("unidades_vendidas") > 0),
                F.col("categoria_analitica")
            )
        ).alias("categorias_activas"),

        F.sum(
            F.when(
                (F.col("categoria_analitica") != "Sin categoría") &
                (F.col("primer_mes_venta") == F.col("mes")),
                1
            ).otherwise(0)
        ).alias("categorias_nuevas"),

        F.sum(
            F.when(
                (F.col("categoria_analitica") != "Sin categoría") &
                F.col("mes_venta_anterior").isNotNull() &
                (
                    F.months_between(
                        F.col("mes"),
                        F.col("mes_venta_anterior")
                    ) > 1
                ),
                1
            ).otherwise(0)
        ).alias("categorias_reactivadas"),

        F.round(
            100 * F.sum(
                F.when(
                    F.col("ranking_categoria") <= 5,
                    F.col("gmv_categoria_brl")
                ).otherwise(0)
            ) / F.max("gmv_total_mes"),
            2
        ).alias("participacion_top5_pct"),

        F.round(
            100 * F.sum(
                F.when(
                    F.col("ranking_categoria") <= 10,
                    F.col("gmv_categoria_brl")
                ).otherwise(0)
            ) / F.max("gmv_total_mes"),
            2
        ).alias("participacion_top10_pct"),

        F.round(
            10000 * F.sum(
                F.pow(
                    F.col("gmv_categoria_brl") /
                    F.col("gmv_total_mes"),
                    2
                )
            ),
            2
        ).alias("hhi_portfolio"),

        F.max(
            F.when(
                F.col("ranking_categoria") == 1,
                F.col("categoria_analitica")
            )
        ).alias("categoria_lider"),

        F.round(
            100 * F.max(
                F.when(
                    F.col("ranking_categoria") == 1,
                    F.col("gmv_categoria_brl")
                )
            ) / F.max("gmv_total_mes"),
            2
        ).alias("participacion_lider_pct")
    )
)

# 5. Hash determinístico
columnas_hash = [
    c for c in portfolio.columns
    if c != "gold_row_hash"
]

portfolio = portfolio.withColumn(
    "gold_row_hash",
    F.sha2(
        F.to_json(
            F.struct(*[
                F.col(c) for c in columnas_hash
            ]),
            options={"ignoreNullFields": "false"}
        ),
        256
    )
)

# 6. Publicación Delta con el MISMO MERGE que las seis tablas anteriores.
# No sobrescribe una Gold idéntica y mantiene estable el hash publicado.
publish_gold(portfolio, "gold_monthly_portfolio_evolution", ["mes"])

print(f"PUBLICADO: {TARGET}")
print("Grano: una fila por mes")

display(
    spark.table(TARGET).orderBy("mes")
)

In [0]:
from pyspark.sql import functions as F

BASE = "workspace.olist_lakehouse"

portfolio = spark.table(
    f"{BASE}.gold_monthly_portfolio_evolution"
)

ventas = spark.table(
    f"{BASE}.gold_monthly_sales"
)

categorias = spark.table(
    f"{BASE}.gold_monthly_category_share"
)

# 1. Una fila por mes
assert portfolio.count() == portfolio.select("mes").distinct().count(), \
    "ERROR: meses duplicados"

# 2. Mismos meses que ventas
assert portfolio.select("mes").exceptAll(
    ventas.select("mes")
).count() == 0, "ERROR: meses distintos respecto de ventas"

# 3. Reconciliación de GMV
comparacion = (
    portfolio.alias("p")
    .join(ventas.alias("v"), on="mes")
    .select(
        "mes",
        F.abs(
            F.col("p.gmv_brl") - F.col("v.gmv_brl")
        ).alias("diferencia")
    )
)

assert comparacion.filter(
    F.col("diferencia") > 0.01
).count() == 0, "ERROR: GMV no coincide"

# 4. Porcentajes válidos
assert portfolio.filter(
    (F.col("participacion_top5_pct") < 0) |
    (F.col("participacion_top5_pct") > 100) |
    (F.col("participacion_top10_pct") < 0) |
    (F.col("participacion_top10_pct") > 100) |
    (F.col("participacion_top5_pct") >
     F.col("participacion_top10_pct")) |
    (F.col("hhi_portfolio") < 0) |
    (F.col("hhi_portfolio") > 10000)
).count() == 0, "ERROR: indicadores fuera de rango"

# 5. Categorías nuevas no pueden superar las activas
assert portfolio.filter(
    F.col("categorias_nuevas") > F.col("categorias_activas")
).count() == 0, "ERROR: categorías nuevas inconsistentes"

print("VALIDACIONES PORTFOLIO SUPERADAS")
print("Meses:", portfolio.count())

display(portfolio.orderBy("mes"))